# P3b -- Label-efficiency curves, flood/SAR task (25%/50% budgets)

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of P2/P3a (both done 2026-10-02/03). Completes Phase P3 (both tasks).
Prithvi is excluded here -- verified inapplicable in P2 (no SAR-native
Sen1Floods11-pretrained checkpoint exists publicly, and optical imagery is
unusable for this flood window: 70-99.8% cloud cover on every available scene).
Only U-Net and Clay v1.5 are compared. See
`Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md` for
the full track.

**Real sample-size caveat, worth stating up front, not discovered after the fact:**
P2's AOI has only 36 tiles total (30 train, 6 held-out). 25%/50% budgets here are
nested prefixes of a fixed-seed shuffle (seed 42, same convention as P3a) of those 30
train tiles: 8 tiles (25%, `round(30*0.25)=8`) and 15 tiles (50%). An 8-tile training
set is a genuinely tiny label-efficiency budget in absolute terms -- flagged honestly,
not something to read more into than it is. 100% reuses P2's already-verified numbers
(U-Net IoU 0.130, Clay IoU 0.040), not re-run.

**Robustness:** each (model, budget) run is wrapped so a late failure doesn't lose
earlier results -- the comparison CSV is written incrementally after every single run.


## 1. Setup

In [ ]:
!pip install -q geopandas rasterio rasterstats shapely segmentation-models-pytorch lightning torchmetrics huggingface_hub pystac-client planetary-computer stackstac rioxarray
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline** -- same cheap-insurance
reason as every earlier phase in this track, even though this notebook doesn't use
`terratorch` (the original numpy-upgrade trigger).

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p3b.py
import sys, traceback
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import numpy as np
import torch
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassJaccardIndex

from src.imagery import get_imagery
from src.sar_flood import db_from_power
from src.landcover import WATER, reclassify_worldcover
from src.tiling import assign_tile_split, tile_array

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

AOI = (91.15, 24.5, 91.5, 24.8)  # same Dirai, Sunamganj AOI as H5/P2
BEFORE_DATE = '2026-06-19'

# ---- same real data pipeline as P2's U-Net section, exactly ----
before, before_meta = get_imagery(AOI, f'{BEFORE_DATE}/{BEFORE_DATE}', sensor='sentinel-1', resolution=20, max_scenes=1)
before_vv_db = db_from_power(before.sel(band='vv').values)

import pystac_client, planetary_computer, stackstac
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=20, epsg=before_meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
water_label_full = (reclassify_worldcover(worldcover) == WATER).astype('int64')

common_h = min(before.shape[1], water_label_full.shape[0])
common_w = min(before.shape[2], water_label_full.shape[1])
sar_features = np.nan_to_num(np.stack([before_vv_db, db_from_power(before.sel(band='vh').values)]))[:, :common_h, :common_w]
water_label = water_label_full[:common_h, :common_w]

TILE_SIZE = 256
feat_tiles, coords = tile_array(sar_features, tile_size=TILE_SIZE)
label_tiles, _ = tile_array(water_label[None], tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=sar_features.shape[-1], test_fraction=0.3)

all_train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
all_train_labels = [l[0] for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l[0] for l, t in zip(label_tiles, test_flags) if t]
N_TRAIN = len(all_train_feats)
print(f'{N_TRAIN} train tiles, {len(test_feats)} held-out test tiles (same split as P2)')

if N_TRAIN < 4 or len(test_feats) < 1:
    print('too few tiles for a meaningful label-efficiency curve in this AOI -- same real limitation P2/H5 found')
    sys.exit(0)

rng = np.random.RandomState(42)  # same seed/convention as P3a
shuffled_idx = rng.permutation(N_TRAIN)

results = []

def save_results():
    pd.DataFrame(results).to_csv('comparison_p3b.csv', index=False)
    print('--- comparison_p3b.csv so far ---')
    print(pd.DataFrame(results).to_string(index=False))

# Already-verified 100% results from P2 -- not re-run.
results.append({'model': 'sar_unet_fresh_gpu', 'budget': 1.0, 'iou_water': 0.130})
results.append({'model': 'clay_v1.5_sar', 'budget': 1.0, 'iou_water': 0.040})
save_results()

CLAY_SAR_MEAN = np.array([-12.113, -18.673], dtype='float32')
CLAY_SAR_STD = np.array([8.314, 8.017], dtype='float32')
CLAY_SAR_WAVES = torch.tensor([3.5, 4.0], dtype=torch.float32)
CLAY_SAR_GSD = torch.tensor(20.0)


def run_unet(train_feats, train_labels, budget):
    feat_mean = np.stack(train_feats).mean(axis=(0, 2, 3), keepdims=True)[0]
    feat_std = np.stack(train_feats).std(axis=(0, 2, 3), keepdims=True)[0] + 1e-6

    class SarDataset(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            f = (self.feats[idx] - feat_mean) / feat_std
            return torch.from_numpy(f).float(), torch.from_numpy(self.labels[idx][None]).float()

    train_loader = DataLoader(SarDataset(train_feats, train_labels), batch_size=4, shuffle=True, num_workers=2)
    test_loader = DataLoader(SarDataset(test_feats, test_labels), batch_size=4, num_workers=2)

    class SarUNet(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.model = smp.Unet(encoder_name='resnet18', encoder_weights=None, in_channels=2, classes=1)
            self.dice = smp.losses.DiceLoss(mode='binary')
            self.bce = torch.nn.BCEWithLogitsLoss()
        def forward(self, x):
            return self.model(x)
        def _step(self, batch, stage):
            x, y = batch
            logits = self(x)
            loss = self.dice(logits, y) + self.bce(logits, y)
            self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
            return logits, y, loss
        def training_step(self, batch, _):
            _, _, loss = self._step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._step(batch, 'val')
        def test_step(self, batch, _):
            logits, y, _ = self._step(batch, 'test')
            preds = torch.sigmoid(logits) > 0.5
            true = y > 0.5
            inter = (preds & true).sum().float()
            union = (preds | true).sum().float()
            iou = inter / union if union > 0 else torch.tensor(1.0, device=preds.device)
            self.log('test/iou_water', iou)
        def configure_optimizers(self):
            return torch.optim.Adam(self.parameters(), lr=1e-3)

    model = SarUNet()
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100, log_every_n_steps=5,
                          callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p3b_unet_{budget}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return {'model': 'sar_unet_fresh_gpu', 'budget': budget, 'iou_water': round(metrics[0]['test/iou_water'], 3)}


def run_clay(train_feats, train_labels, budget):
    class ClayDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] - CLAY_SAR_MEAN[:, None, None]) / CLAY_SAR_STD[:, None, None]
            return {'pixels': torch.from_numpy(x.astype('float32')), 'label': torch.from_numpy(self.labels[idx]),
                    'time': torch.zeros(4), 'latlon': torch.zeros(4)}

    train_loader = DataLoader(ClayDS(train_feats, train_labels), batch_size=4, shuffle=True, num_workers=2)
    test_loader = DataLoader(ClayDS(test_feats, test_labels), batch_size=4, num_workers=2)

    import huggingface_hub
    ckpt_path = huggingface_hub.hf_hub_download(repo_id='made-with-clay/Clay', filename='v1.5/clay-v1.5.ckpt')

    class ClaySegmentor(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.model = Segmentor(num_classes=2, ckpt_path=None)
            self.model.encoder.load_from_ckpt(ckpt_path)
            self.loss_fn = smp.losses.FocalLoss(mode='multiclass')
            self.iou_per_class = MulticlassJaccardIndex(num_classes=2, average='none')
        def forward(self, batch):
            return self.model({'pixels': batch['pixels'], 'time': batch['time'], 'latlon': batch['latlon'],
                                'gsd': CLAY_SAR_GSD, 'waves': CLAY_SAR_WAVES})
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad],
                                           lr=1e-5, weight_decay=0.05, betas=(0.9, 0.95))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=1, eta_min=1e-5 * 100)
            return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}
        def _shared_step(self, batch, stage):
            labels = batch['label'].long()
            outputs = self(batch)
            if outputs.shape[-2:] != labels.shape[-2:]:
                outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
            loss = self.loss_fn(outputs, labels)
            self.log(f'{stage}/loss', loss, on_epoch=True, prog_bar=(stage != 'train'))
            return outputs, labels, loss
        def training_step(self, batch, _):
            _, _, loss = self._shared_step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._shared_step(batch, 'val')
        def test_step(self, batch, _):
            outputs, labels, _ = self._shared_step(batch, 'test')
            per_class = self.iou_per_class(outputs, labels)
            self.log('test/iou_not_water', per_class[0], on_epoch=True)
            self.log('test/iou_water', per_class[1], on_epoch=True)

    model = ClaySegmentor()
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='bf16-mixed', max_epochs=100, log_every_n_steps=5,
                          callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p3b_clay_{budget}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return {'model': 'clay_v1.5_sar', 'budget': budget, 'iou_water': round(metrics[0]['test/iou_water'], 3)}


for budget in [0.25, 0.50]:
    n = max(1, int(round(N_TRAIN * budget)))
    subset_idx = shuffled_idx[:n]
    sub_feats = [all_train_feats[i] for i in subset_idx]
    sub_labels = [all_train_labels[i] for i in subset_idx]
    print(f'=== budget {budget} -> {n}/{N_TRAIN} train tiles ===')

    for name, fn in [('unet', run_unet), ('clay', run_clay)]:
        try:
            print(f'--- running {name} @ budget {budget} ---')
            row = fn(sub_feats, sub_labels, budget)
            print(row)
            results.append(row)
        except Exception as e:
            print(f'FAILED: {name} @ budget {budget}: {e}')
            traceback.print_exc()
            results.append({'model': name, 'budget': budget, 'error': str(e)})
        save_results()

print('=== FINAL ===')
print(pd.DataFrame(results).to_string(index=False))


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p3b.py'], capture_output=True, text=True)
print(proc.stdout[-25000:])
print('---stderr (tail)---')
print(proc.stderr[-10000:])
print('returncode:', proc.returncode)
# NOTE: same as P3a -- no assert on returncode==0. A single (model, budget) failure
# is caught and logged inside run_p3b.py itself, so a partial comparison_p3b.csv is a
# valid, honest outcome worth reading, not a reason to discard everything.


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p3b.csv')
print(comparison.to_string(index=False))


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** Reuses P2's already-
  verified data pipeline and model recipes (U-Net, Clay) exactly -- the only new thing
  here is subsetting the 30 train tiles by budget, so Kaggle-specific friction on first
  run should be minimal, but is still expected and gets fixed from the real traceback
  if it shows up, not guessed at.
- **8 training tiles at the 25% budget is a genuinely tiny number.** This whole
  notebook's results should be read as "what happens with very little flood-labeled
  data in one small AOI," not as a general statement about SAR label efficiency --
  stated here, not left for the reader to discover by noticing a suspiciously extreme
  result.
- Once this runs clean (or partially clean): bring back the actual numbers, including
  any 'error' rows, and note whether either model's curve at 8/15/30 tiles looks like
  a real trend or just noise at this sample size -- n this small may simply not support
  a meaningful label-efficiency conclusion, and saying so honestly is a valid finding.


**Update 2026-10-03, real result (Kaggle kernel version 1, COMPLETE, ~24 min, T4 GPU,
all 4 new runs succeeded with zero failures):**

| model | budget (tiles) | held-out IoU (water) |
|---|---|---|
| sar_unet_fresh_gpu | 0.25 (8) | 0.105 |
| sar_unet_fresh_gpu | 0.50 (15) | 0.092 |
| sar_unet_fresh_gpu | 1.00 (30) | 0.130 |
| clay_v1.5_sar | 0.25 (8) | 0.078 |
| clay_v1.5_sar | 0.50 (15) | 0.074 |
| clay_v1.5_sar | 1.00 (30) | 0.040 |

**Neither model shows an interpretable label-efficiency curve here -- the honest
finding this notebook's own notes anticipated as a real possibility.** U-Net is
non-monotonic (0.105 -> 0.092 -> 0.130, dips at 50% before rising at 100%). Clay
actually gets *worse* with more data (0.078 -> 0.074 -> 0.040), the opposite of what
P3a found for the land-cover task. At 8-30 total training tiles, this is almost
certainly dominated by sampling noise rather than a real relationship between label
budget and performance -- there is simply not enough data in this one small AOI to
support a meaningful label-efficiency conclusion for the flood/SAR task. This is
itself the real, honest answer, not a null result to be hidden: P3's design (reusing
H5/P2's real but tiny event-specific AOI) cannot resolve the question it set out to
ask for this task, unlike the land-cover task (P3a, 64 tiles), where a real trend was
visible. Confirmed reproducing the exact same 30 train / 6 held-out split as P2
(printed directly, not assumed) before training anything, so this is not a data
pipeline mismatch -- just too little real data for this specific event.
